
---

# ***`Predict Using Sklearn Pipeline`***

---

- The main goal of this notebook is to implement the `scikit-learn pipeline` without worring about the accuracy to build better concepts.

Pipleline Will consist of following Steps

1. Imputation of Missing Values
2. OneHotEncode the `Sex` and `Embarked` Column
3. Scale the Numerical Columns
4. Feature Selection (Selec Best 5 Features) # optional
5. Train the model Using the Decision tree Classifier.


---

### **Import Libraries**

In [19]:
import os
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV
)
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    OneHotEncoder,
    LabelEncoder,
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline , make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest,chi2

import warnings
warnings.filterwarnings("ignore")

### **Load the Dataset**

In [2]:
df = pd.read_csv(r"https://raw.githubusercontent.com/campusx-official/100-days-of-machine-learning/refs/heads/main/day29-sklearn-pipelines/train.csv")

df.sample(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
157,158,0,3,"Corn, Mr. Harry",male,30.0,0,0,SOTON/OQ 392090,8.0500,NaN,S
466,467,0,2,"Campbell, Mr. William",male,NaN,0,0,239853,0.0000,NaN,S
808,809,0,2,"Meyer, Mr. August",male,39.0,0,0,248723,13.0000,NaN,S
530,531,1,2,"Quick, Miss. Phyllis May",female,2.0,1,1,26360,26.0000,NaN,S
609,610,1,1,"Shutes, Miss. Elizabeth W",female,40.0,0,0,PC 17582,153.4625,C125,S
505,506,0,1,"Penasco y Castellana, Mr. Victor de Satode",male,18.0,1,0,PC 17758,108.9000,C65,C
424,425,0,3,"Rosblom, Mr. Viktor Richard",male,18.0,1,1,370129,20.2125,NaN,S
290,291,1,1,"Barber, Miss. Ellen ""Nellie""",female,26.0,0,0,19877,78.8500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
330,331,1,3,"McCoy, Miss. Agnes",female,NaN,2,0,367226,23.2500,NaN,Q


### **Drop Unnecessory Columns**

- Since these columns do not play important part in my analysis.

In [3]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

### **Train test Split the Data**

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=['Survived']),
    df['Survived'],
    test_size=0.2,
    random_state=42,
)

In [5]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


### **Imputation Transformer**

In [6]:
# Create a ColumnTransformer to apply different preprocessing steps to specific columns of the dataset.
# The ColumnTransformer allows for different transformers to be applied to different columns of the input data.

trf1 = ColumnTransformer([
    # First transformer: 'impute_age'
    # This uses a SimpleImputer to fill in missing values for the 'Age' feature (assumed to be at index 2).
    ('impute_age', SimpleImputer(), [2]),

    # Second transformer: 'impute_embarked'
    # This uses a SimpleImputer with the strategy set to "most_frequent" to fill in missing values for the 'Embarked' feature (assumed to be at index 6).
    ('impute_embarked', SimpleImputer(strategy="most_frequent"), [6]),
], 
# The 'remainder' parameter specifies what to do with the remaining columns not explicitly transformed.
# Here, 'passthrough' means that the other columns will be passed through unchanged.
remainder='passthrough')

### **One Hot Encoding**

In [7]:
# Create a ColumnTransformer to apply one-hot encoding to specific categorical columns in the dataset.
# The ColumnTransformer allows for different transformers to be applied to different columns of the input data.

trf2 = ColumnTransformer([
    # Transformer: 'ohe_sex_embarked'
    # This uses a OneHotEncoder to convert categorical variables into a one-hot encoded format.
    # The 'sparse=False' parameter indicates that the output should be a dense array rather than a sparse matrix.
    # The 'handle_unknown' parameter set to 'ignore' ensures that if there are unknown categories during transformation,
    # they will not cause errors; instead, they will be ignored.
    ('ohe_sex_embarked', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), [1, 6]),  # Apply to columns 1 and 6
], 
# The 'remainder' parameter specifies what to do with the remaining columns not explicitly transformed.
# Here, 'passthrough' means that the other columns will be passed through unchanged.
remainder='passthrough')

### **Scaling**

In [8]:
# Create a ColumnTransformer to apply scaling to a specific range of columns in the dataset.
# The ColumnTransformer allows for different transformers to be applied to different columns of the input data.

trf3 = ColumnTransformer([
    # Transformer: 'scale'
    # This uses a MinMaxScaler to scale the features to a range between 0 and 1.
    # The 'slice(0, 10)' indicates that the transformation will be applied to columns 0 through 9 (the first 10 columns).
    ("scale", MinMaxScaler(), slice(0, 10))
])

# Note: The MinMaxScaler is useful for normalizing features when the scales are different, ensuring they contribute equally to model training.

### **Feature Selection**

In [9]:
# Create a SelectKBest object to select the top k features based on a statistical test.
# In this case, we are using the chi-squared statistic to evaluate the features.

trf4 = SelectKBest(score_func=chi2, k=8)  # Select the top 8 features based on the chi-squared score

# Note:
# - The 'score_func' parameter specifies the function to use for scoring the features.
# - 'chi2' is suitable for non-negative feature values (e.g., counts or frequencies).
# - The 'k' parameter determines the number of top features to select, which is set to 8 in this case.

### **Train Decision Tree Classifier Model**

In [10]:
trf5 = DecisionTreeClassifier()

### **Create a Pipeline**

In [11]:
pipe = Pipeline([
    ('trf1',trf1),
    ('trf2',trf2),
    ('trf3',trf3),
    ('trf4',trf4),
    ('trf5',trf5),
])

In [12]:
# # alternate easy approach
# pipe = make_pipeline(trf1,trf2,trf3,trf4,trf5)

#### **Train the Pipeline**

In [13]:
pipe.fit(X_train,y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('trf3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('trf4',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x000001E2430C09A0>)),
                ('trf5', DecisionTreeClassifier())])

#### **Explore the Pipeline**

In [14]:
pipe.named_steps

{'trf1': ColumnTransformer(remainder='passthrough',
                   transformers=[('impute_age', SimpleImputer(), [2]),
                                 ('impute_embarked',
                                  SimpleImputer(strategy='most_frequent'),
                                  [6])]),
 'trf2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_sex_embarked',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [1, 6])]),
 'trf3': ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0, 10, None))]),
 'trf4': SelectKBest(k=8, score_func=<function chi2 at 0x000001E2430C09A0>),
 'trf5': DecisionTreeClassifier()}

### **Predict** 

In [15]:
y_pred = pipe.predict(X_test)

In [16]:
print(f"Accuracy Score: {accuracy_score(y_test,y_pred)}")

Accuracy Score: 0.6256983240223464


### **Cross Validation Using Pipeline**

In [18]:
cross_val_score(pipe,X_train,y_train,cv=5,scoring='accuracy').mean()

0.6391214419383433

### **Grid Search Using Pipeline**

In [20]:
params = {
    'trf5__max_depth' : [1,2,3,4,5,None]
}

In [21]:
grid = GridSearchCV(pipe,params,cv=5,scoring='accuracy')
grid.fit(X_train,y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('trf1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('impute_age',
                                                                         SimpleImputer(),
                                                                         [2]),
                                                                        ('impute_embarked',
                                                                         SimpleImputer(strategy='most_frequent'),
                                                                         [6])])),
                                       ('trf2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('ohe_sex_embarked',
                                                                         OneHotEncoder(handle_unknown='ignore',
                                                                                       sparse_output=False),
                                                                         [1,
                                                                          6])])),
                                       ('trf3',
                                        ColumnTransformer(transformers=[('scale',
                                                                         MinMaxScaler(),
                                                                         slice(0, 10, None))])),
                                       ('trf4',
                                        SelectKBest(k=8,
                                                    score_func=<function chi2 at 0x000001E2430C09A0>)),
                                       ('trf5', DecisionTreeClassifier())]),
             param_grid={'trf5__max_depth': [1, 2, 3, 4, 5, None]},
             scoring='accuracy')

In [22]:
grid.best_score_

0.6391214419383433

In [23]:
grid.best_params_

{'trf5__max_depth': 2}

### **Exporting the Pipeline**

In [25]:
pickle.dump(pipe,open('models/pipe.pkl', 'wb'))

### **Load the Exported Pipeline**

In [26]:
pipe = pickle.load(open('models/pipe.pkl', 'rb'))

In [ ]:
# Assume a user input for a single data instance representing a passenger's details.
# The input includes various features such as passenger class, gender, age, number of siblings/spouses aboard,
# number of parents/children aboard, fare paid, and embarkation point.

test_input_2 = np.array([2, 'male', 31.0, 0, 0, 10.5, 'S'], dtype=object).reshape(1, 7)

# Breakdown of the input features:
# - 2: Passenger class (assumed to be an integer)
# - 'male': Gender of the passenger
# - 31.0: Age of the passenger (float)
# - 0: Number of siblings/spouses aboard (integer)
# - 0: Number of parents/children aboard (integer)
# - 10.5: Fare paid (float)
# - 'S': Embarkation point (categorical, e.g., Southampton)

# The 'dtype=object' allows for mixed data types within the array.
# The 'reshape(1, 7)' method reshapes the array into a 2D format with 1 row and 7 columns, suitable for further processing.

In [28]:
pipe.predict(test_input_2)

array([0], dtype=int64)

---